# 第 10 集 — 看见声音,再回到 Max

⏱ 约 25 分钟 · 难度 ★★☆

**这一集控制的音乐元素:没有新的。** 这一集是把 Python 里做的东西 **看清楚** 并 **带回 Max**。

**Max 里的对应:** `spectroscope~`(看),`buffer~` 的 `read`(读 wavetable),`dict` 的 `read`(读 preset)。

**你会听到:** 第 8 集的三个 preset — 这次同时看着它们的 spectrogram 听。

**你会得到两个文件:**
- `export/wavetable.wav`:16 个 frame 的 wavetable
- `export/presets.json`:三个 preset

**学完你能做到:**
- 读懂 spectrum 和 spectrogram
- 从 spectrogram 上认出 filter envelope、pitch 变化、wavetable morph
- 把 wavetable 和 preset 导出成 Max 能读的文件

## 工具箱

前几集做好的东西,原样搬过来。运行一下就行,不用重读(难的地方都有 `#` 注释)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip 把超过 ±1 的部分压住,防止爆音
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # 只画开头这么多个 sample
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

import os
import json
from scipy.io import wavfile

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # 表里每个位置的 phase,0 到 1
    table = np.zeros(N)                        # 先做一张全是 0 的空表
    for k, amp in enumerate(harmonic_amps, start=1):
        # 第 k 个 harmonic:在表里走 k 个周期,音量 amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize:峰值变成 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # 第 f 张表有几个 harmonic:从 1 个均匀增加到 max_harmonics 个
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # 叠成二维 array:F 行 × N 列

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- 在哪两张表之间 ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # 0..1 换算成 0..F-1 的 "表编号",可以带小数
    f0 = np.floor(fpos).astype(int)            # 下面那张表
    f1 = np.minimum(f0 + 1, F - 1)             # 上面那张表(不超过最后一张)
    ffrac = fpos - f0                          # 离下面那张有多远,0..1

    # --- 在表里的哪两个位置之间(和第 3 集一样)---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # 左边的位置
    i1 = (i0 + 1) % N                          # 右边的位置,到表尾绕回 0
    frac = p - np.floor(p)                     # 小数部分

    # 先在每张表内部插值,再在两张表之间插值
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # 时间不能是 0,否则下面会除以 0
    # 按住期间:(0,0) → (a,1) → (a+d,s),之后停在 s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # 松开那一刻 envelope 有多高(可能还没走完 attack)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # 从那个高度用 r 秒直线降到 0;np.clip 让它降到 0 后不再变负
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # 松开前用 held,松开后用 released

def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff 给一个数或一个 array 都行:统一变成和声音一样长的 array
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- 把 cutoff / res 换算成 filter 内部用的系数(每个 sample 一组)---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping:res 越大 k 越小,共鸣越强
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() 把 array 变成普通 list:在循环里逐个取数时快很多
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # 两个状态变量 = gen~ 里的两个 history

    for n in range(n_samples):                 # 一个 sample 一个 sample 按顺序算
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 就是 lowpass 的输出
        ic1 = 2 * v1 - ic1                     # 更新状态,留给下一个 sample 用
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

def phasor(freq, n_samples):
    # freq / sr = 这一个 sample 里 phase 往前走多少
    # cumsum 把每一步累加起来;% 1 只留小数部分
    return np.cumsum(np.broadcast_to(freq, n_samples) / sr) % 1

def mtof(m):
    return 440 * 2 ** ((m - 69) / 12)          # MIDI 69 = 440 Hz;每 12 个半音翻一倍

def synth(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr

    # 两个 envelope
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    mod_env = adsr(t, p["mod_a"], p["mod_d"], p["mod_s"], p["mod_r"], gate)

    # oscillator:wt_pos 旋钮 + mod env 推上去的量
    phase = phasor(mtof(p["note"]), n)
    x = osc_wavetable(frames, phase, p["wt_pos"] + p["wt_env"] * mod_env)

    # filter:cutoff 旋钮,被 mod env 往上推 cut_env 个八度
    x = lowpass(x, p["cutoff"] * 2 ** (p["cut_env"] * mod_env), p["res"])

    return x * amp_env                         # VCA

presets = {
    "pluck": dict(note=45, wt_pos=0.6, wt_env=0.0, cutoff=300, res=0.2, cut_env=4.0,
                  amp_a=0.005, amp_d=0.30, amp_s=0.0, amp_r=0.10,
                  mod_a=0.001, mod_d=0.15, mod_s=0.0, mod_r=0.10),
    "pad":   dict(note=57, wt_pos=0.2, wt_env=0.5, cutoff=800, res=0.1, cut_env=2.0,
                  amp_a=0.60, amp_d=0.30, amp_s=0.8, amp_r=0.70,
                  mod_a=1.00, mod_d=0.50, mod_s=0.6, mod_r=0.50),
    "wow":   dict(note=33, wt_pos=0.3, wt_env=0.7, cutoff=150, res=0.6, cut_env=4.5,
                  amp_a=0.01, amp_d=0.10, amp_s=0.9, amp_r=0.10,
                  mod_a=0.35, mod_d=0.40, mod_s=0.2, mod_r=0.20),
}

frames = make_frames()

n = sr                                         # 1 秒
t = np.arange(n) / sr

## 1. Spectrum:声音的 "成分表"

波形图(`scope~`)告诉你 **每一刻的振幅**。它回答不了 "这个声音亮不亮"。

spectrum(`spectroscope~`)换一个角度:**这个声音里有哪些频率,各有多响。**

把声音变成 spectrum 的运算叫 FFT。这里不讲它怎么算,只讲怎么读。下面的函数第 6 集见过。

- 横轴:频率(Hz)
- 纵轴:响度(dB)。0 = 最响的那个频率;每低 20 dB,振幅变成十分之一

### 一个 sine

**先猜:** 一个 440 Hz 的 sine,spectrum 长什么样?

In [ ]:
def spectrum(x):
    X = np.abs(np.fft.rfft(x * np.hanning(len(x))))        # 每个频率的强度
    f = np.fft.rfftfreq(len(x), 1 / sr)                    # 对应的频率刻度(Hz)
    return f, 20 * np.log10(X / np.max(X) + 1e-9)          # 换成 dB,最响的是 0

def spec_plot(signals, max_freq=5000):
    plt.figure(figsize=(9, 3.5))
    for name in signals:
        f, db = spectrum(signals[name])
        plt.plot(f, db, label=name, linewidth=0.8)
    plt.xlim(0, max_freq)
    plt.ylim(-90, 5)
    plt.xlabel("frequency (Hz)")
    plt.ylabel("level (dB)")
    plt.legend()
    plt.show()

sine440 = np.sin(2 * np.pi * 440 * t)
spec_plot({"sine 440 Hz": sine440}, max_freq=2000)

答案:一根竖线,在 440 Hz。sine 只包含一个频率 — 这就是它听起来 "干净" 的原因。

### 两个 sine 加起来

第 1 集做过和弦:把 sine 加起来。

**应该看到:** 两根竖线,在 440 和 660 Hz。660 那根矮 6 dB,因为它的振幅是一半。

In [ ]:
two = np.sin(2 * np.pi * 440 * t) + 0.5 * np.sin(2 * np.pi * 660 * t)
spec_plot({"440 Hz + 660 Hz at half level": two}, max_freq=2000)

### Wavetable 的 frame

第 3 集的表是 "叠 harmonic" 做的。spectrum 把这些 harmonic 重新摊开给你看。

**应该看到:**
- frame 0:一根线(110 Hz)
- frame 15:32 根等间距的线(110、220、330 …… 到 3520 Hz),越往右越矮

**"暗" 和 "亮" 在图上就是:竖线少 / 竖线多。**

In [ ]:
phase = phasor(110, n)

spec_plot({
    "frame 15 (wt_pos = 1)": osc_wavetable(frames, phase, 1.0),
    "frame 0 (wt_pos = 0)": osc_wavetable(frames, phase, 0.0),
})

### Filter 做了什么

**应该看到:** 过了 lowpass 之后,同样那一排竖线,600 Hz 以上的被压低了,而 600 Hz 那里因为 resonance 顶起了一个峰。

In [ ]:
raw = osc_wavetable(frames, phase, 1.0)

spec_plot({
    "raw": raw,
    "lowpass 600 Hz, res 0.8": lowpass(raw, 600, 0.8),
}, max_freq=4000)

## 2. Spectrogram:会动的 spectrum

spectrum 是一张 "照片":它把整段声音压成一张图,看不出 **什么时候** 发生了什么。

spectrogram 是 "录像":把声音切成很多小段,每段算一次 spectrum,竖着排起来。

- 横轴:时间
- 纵轴:频率
- 颜色:越亮越响

### 一个上滑的 sine

**先猜:** 一个从 100 Hz 滑到 4000 Hz 的 sine,spectrogram 长什么样?

In [ ]:
def spectrogram(x, title="", max_freq=4000):
    plt.figure(figsize=(9, 3.5))
    # NFFT = 每一小段多长;noverlap = 相邻两段重叠多少
    with np.errstate(divide="ignore"):         # 完全静音的地方算 dB 会警告,这里忽略它
        plt.specgram(x + 1e-9, NFFT=2048, Fs=sr, noverlap=1536, cmap="magma", vmin=-120)
    plt.ylim(0, max_freq)
    plt.xlabel("time (s)")
    plt.ylabel("frequency (Hz)")
    plt.title(title)
    plt.show()

glide = np.sin(2 * np.pi * phasor(np.geomspace(100, 4000, n), n))

spectrogram(glide, "sine gliding 100 -> 4000 Hz")
play(glide)

答案:一条往上弯的亮线。它是弯的而不是直的,因为我们用了 `geomspace`(按倍数走),而图的纵轴是按 Hz 画的。

### 三个 preset

现在看第 8 集的三个 preset。**先听,再看图,把听到的东西在图上找出来。**

**pluck — 应该看到:** 最左边一条很窄的竖直亮带(开头那一下很亮),然后迅速只剩底部几条线,0.4 秒后全黑。

In [ ]:
x = synth(presets["pluck"])
display(play(x))
spectrogram(x, "pluck")

**pad — 应该看到:** 水平的线从左边慢慢 "亮起来"(淡入),同时亮的区域慢慢往上长(变亮),1.2 秒后整体渐暗。

In [ ]:
x = synth(presets["pad"])
display(play(x))
spectrogram(x, "pad")

**wow — 应该看到:** 底部一排很密的水平线(低音的 harmonic 间距小)。亮的区域先往上鼓起来,到 0.35 秒左右最高,然后落回去。

你听到的 "wow" 就是图上那个鼓包。它的形状就是 mod envelope 的形状。

In [ ]:
x = synth(presets["wow"])
display(play(x))
spectrogram(x, "wow", max_freq=3000)

### 小测验:看图猜参数

下面三个声音是同一个 preset 改了 **一个** 参数得到的。不听,只看图,猜每张图改的是什么:

- (a) `cut_env` 改成 0
- (b) `note` 提高一个八度
- (c) `mod_a` 改成 1.0

In [ ]:
base = presets["wow"]
quiz = {
    "A": dict(base, mod_a=1.0),
    "B": dict(base, cut_env=0.0),
    "C": dict(base, note=base["note"] + 12),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, name in zip(axes, quiz):
    with np.errstate(divide="ignore"):
        ax.specgram(synth(quiz[name]) + 1e-9, NFFT=2048, Fs=sr, noverlap=1536, cmap="magma", vmin=-120)
    ax.set_ylim(0, 3000)
    ax.set_title(name)
    ax.set_xlabel("time (s)")
axes[0].set_ylabel("frequency (Hz)")
plt.show()

<details><summary>答案</summary>

- **A = (c)** `mod_a = 1.0`:鼓包的最高点移到了 1 秒处,上升得很慢。
- **B = (a)** `cut_env = 0`:没有鼓包,亮的区域全程一样高(filter 不动)。
- **C = (b)** 高一个八度:水平线之间的间距变成两倍。

</details>

听一下确认:

In [ ]:
for name in quiz:
    print(name)
    display(play(synth(quiz[name])))

## 3. 把 wavetable 送回 Max

`frames` 是 16 × 2048 的二维 array。wav 文件是一维的。所以要把它 "摊平":frame 0 接 frame 1 接 frame 2……

`.reshape(-1)` 就是摊平。先看一个小例子:

In [ ]:
m = np.array([[1, 2, 3],
              [10, 20, 30]])

print("2-D:", m.shape)
print(m)
print("flattened:", m.reshape(-1))             # 一行接一行

这是 wavetable 文件最常见的排法:每个 frame 2048 个 sample,首尾相接。Serum 这类 wavetable synth 可以按这个格式导入。

文件会写到 notebook 旁边的 `export/` 文件夹。

In [ ]:
os.makedirs("export", exist_ok=True)

flat = frames.reshape(-1).astype(np.float32)   # 16 × 2048 → 32768 个 sample 的一条
wavfile.write("export/wavetable.wav", sr, 0.9 * flat)

print("frames:", frames.shape[0])
print("samples per frame:", frames.shape[1])
print("total samples in file:", len(flat))

### 验证

读回来,取出第 5 个 frame(sample 5 × 2048 到 6 × 2048),和原来的 `frames[5]` 比较。

In [ ]:
file_sr, loaded = wavfile.read("export/wavetable.wav")

k = 5
N = frames.shape[1]
frame_from_file = loaded[k * N : (k + 1) * N]          # 第 k 个 frame 在文件里的位置

print("frame 5 matches:", np.allclose(frame_from_file, 0.9 * frames[k], atol=1e-6))

### 在 Max 里怎么用

1. `buffer~ wt` → 发送 `read wavetable.wav`
2. `phasor~` 接到 `wave~ wt` 的第 1 个 inlet
3. `wave~` 的第 2、3 个 inlet 是读取范围的起点和终点,**单位是毫秒**

所以需要把 "第 k 个 frame" 换算成毫秒:

- 起点 = `k × 2048 / sr × 1000`
- 终点 = `(k + 1) × 2048 / sr × 1000`

下面打印出每个 frame 的起止毫秒数,可以直接抄进 Max。要在两个 frame 之间 crossfade,就用两个 `wave~` 读相邻的两段,再用第 4 集的 `(1 - mix) * a + mix * b`(Max 里是 `xfade~` 或一对 `*~`)。

In [ ]:
for k in range(frames.shape[0]):
    start_ms = k * N / sr * 1000
    end_ms = (k + 1) * N / sr * 1000
    print(f"frame {k:2d}:  {start_ms:8.3f} ms  to  {end_ms:8.3f} ms")

## 4. 把 preset 送回 Max

preset 是 `dict`。第 9 集用过 JSON 来存参数。Max 的 `dict` object 可以直接读 JSON 文件。

In [ ]:
with open("export/presets.json", "w") as f:
    json.dump(presets, f, indent=2)

print(json.dumps(presets["wow"], indent=2))    # 文件里 "wow" 那一段长这样

### 在 Max 里怎么用

1. `dict presets` → 发送 `read presets.json`
2. 发送 `get wow::cutoff` → `dict` 输出这个值(150)。`::` 是 "进到里面一层" 的意思
3. 用 `route` 把取出的值分发到 RNBO patch 里同名的 `param` 上

只要 Max 那边的 synth 用 **同样的参数名和同样的 signal flow**(第 8 集那张图),同一个 preset 在两边就应该是同一个声音。

实际做的时候,两边的 filter 和 envelope 曲线不会完全一样(`lores~` 和我们的 `lowpass()` 不是同一个算法),所以听起来会 **很接近但不完全相同**。如果需要完全一致,就要在 gen~ 里把第 6 集的那几行算式原样写一遍。

## 5. 全系列对照表

| 集 | Max / RNBO | Python |
|---|---|---|
| 1 | `cycle~`、`dac~`、`scope~`、signal | `np.sin`、`play`、`scope`、array |
| 2 | `phasor~`、`<~`、`abs~` | `(freq * t) % 1`、`np.where`、`np.abs` |
| 3 | `buffer~`、`index~`、`wave~` | `table[idx]`、线性插值 |
| 4 | WT POS、`xfade~`、`line~` | 二维 `frames`、`np.linspace` |
| 5 | `adsr~`、`function`、`*~` | `np.interp`、`osc * env` |
| 6 | `onepole~`、`lores~`、gen~ `history`、`noise~` | `for` 循环 + 状态变量 |
| 7 | signal 接参数 inlet、LFO、`phasor~` 的 signal 输入 | 参数传 array、`np.cumsum` |
| 8 | RNBO patch、`param`、`dict`、`mtof` | 函数、`dict` |
| 9 | `random`、`sfrecord~`(很别扭) | `rng` + 循环 + 存文件 |
| 10 | `spectroscope~`、`buffer~ read`、`dict read` | `np.fft`、`specgram`、`wavfile`、`json` |

## 三句话带走

1. **Python 里声音是一个 array。** 没有 feedback 的东西整串一起算;有 feedback 的东西(filter、phase 累加)要 "记住上一次"。
2. **参数和 signal 是同一种东西。** 传一个数是静态的,传一个 array 就是 modulation。
3. **Python 负责离线和批量,Max 负责实时和交互。** 用同一套参数名,它们可以互相传东西。

## 练习

**1.** 画出第 2 集的朴素 saw(`2 * phase - 1`)在 2500 Hz 时的 spectrum,横轴到 22050 Hz。找出那些不在 2500 的整数倍上的线 — 那就是 aliasing。

<details><summary>答案</summary>

```python
ph = phasor(2500, n)
spec_plot({"naive saw 2500 Hz": 2 * ph - 1}, max_freq=22050)
```
2500、5000、7500 …… 是真正的 harmonic。夹在它们之间的那些矮线是折回来的。
</details>

**2.** 画第 7 集的 vibrato(220 Hz,每秒 5 次,上下半个半音)的 spectrogram,纵轴只看 0–1500 Hz。**先猜:** 线是什么形状?

<details><summary>答案</summary>

```python
vib = 220 * 2 ** (0.5 / 12 * np.sin(2 * np.pi * 5 * t))
spectrogram(osc_wavetable(frames, phasor(vib, n), 0.8), "vibrato", max_freq=1500)
```
每条 harmonic 都是一条波浪线;越高的 harmonic 波浪幅度越大(因为它们的频率是基音的整数倍,抖动也被放大同样的倍数)。
</details>

**3.** 把第 4 集练习里的 "sine 到 square" frames(或者任何一套你自己的 frames)导出成 `export/my_wavetable.wav`。

<details><summary>答案</summary>

```python
sq = []
for f in range(16):
    amps = []
    for k in range(1, 2 * f + 2):
        amps.append(1 / k if k % 2 == 1 else 0)
    sq.append(make_table(amps))
sq = np.array(sq)
wavfile.write("export/my_wavetable.wav", sr, (0.9 * sq.reshape(-1)).astype(np.float32))
```
</details>

## 接下来可以做什么

- 把 envelope 换成 exponential 曲线(第 5 集第 7 节的思路)
- 加第二个 oscillator、detune、noise
- 用 `sounddevice` 库在 Python 里做实时输出
- 在 gen~ / RNBO 里按第 8 集的 signal flow 重建这个 synth,读入这一集导出的两个文件
- 用第 9 集的数据集训练一个模型:听声音 → 预测参数